In [ ]:
import os
import torch
import numpy as np
from PIL import Image
import pickle

# -------------------------
# LOAD TRAJECTORIES (GT)
# -------------------------
with open("splined_trajectories_3.txt", "rb") as f:
    trajectories = pickle.load(f)

data_root = "trajectory_images"

all_images = []
all_coords = []

for i, traj_folder in enumerate(sorted(os.listdir(data_root))):

    traj_path = os.path.join(data_root, traj_folder)
    if not os.path.isdir(traj_path):
        continue

    frames = []
    traj = np.array(trajectories[i])  # (T,4)

    frame_files = sorted(os.listdir(traj_path))

    for frame_file in frame_files:
        img_path = os.path.join(traj_path, frame_file)

        img = Image.open(img_path).convert("RGB")

        img_np = np.array(img)  # (H, W, 3)

        img_tensor = torch.from_numpy(img_np).float() / 255.0  # ✅ normalize
        img_tensor = img_tensor.permute(2, 0, 1)  # (C,H,W)

        frames.append(img_tensor)

    traj_images = torch.stack(frames)      # (T, C, H, W)
    traj_coords = torch.from_numpy(traj).float()  # (T, 4)

    all_images.append(traj_images)
    all_coords.append(traj_coords)

# stack all
all_images = torch.stack(all_images)  # (N, T, C, H, W)
all_coords = torch.stack(all_coords)  # (N, T, 4)

# save together
torch.save({
    "images": all_images,
    "coords": all_coords
}, "trajectory_dataset.pt")

print("Images shape:", all_images.shape)
print("Coords shape:", all_coords.shape)